# Assessment appeal enrichment with Microsoft Foundry

This is an alternate path for Goal 3. It reads the conformed `dbo.fact_appeal` table created by notebook 3, calls a chat-model deployment through Microsoft Foundry, validates structured AI output, and writes `dbo.fact_appeal_foundry_ai`.

Use either the built-in Fabric AI path in notebook 3 or this Foundry path. You may run both to compare quality. AI output is decision support only and must not approve or reject an appeal.

In [ ]:
%%configure -f
{
  "defaultLakehouse": { "name": "SilverLakehouse" }
}

In [ ]:
variable_library_name = "HackathonVariables"
source_table = "dbo.fact_appeal"
target_table = "dbo.fact_appeal_foundry_ai"
foundry_endpoint_override = ""
foundry_api_key_override = ""
max_appeals = 0
max_workers = 4

## Required variable-library values

The active value set in `HackathonVariables` must provide `foundry_endpoint`, `chat_deployment`, and `api_version`. API-key authentication can additionally use `key_vault_uri` and `foundry_key_secret_name`. Leave both blank to use the notebook user's Entra token.

In [ ]:
import json
import requests
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType
import notebookutils

library = notebookutils.variableLibrary.getLibrary(variable_library_name)
foundry_endpoint = (foundry_endpoint_override or library.getVariable("foundry_endpoint") or "").rstrip("/")
chat_deployment = (library.getVariable("chat_deployment") or "").strip()
api_version = (library.getVariable("api_version") or "").strip()
key_vault_uri = (library.getVariable("key_vault_uri") or "").strip()
foundry_key_secret_name = (library.getVariable("foundry_key_secret_name") or "").strip()

if not foundry_endpoint or not chat_deployment or not api_version:
    raise ValueError("The active variable-library value set must define foundry_endpoint, chat_deployment, and api_version.")

if foundry_api_key_override:
    foundry_api_key = foundry_api_key_override
elif key_vault_uri and foundry_key_secret_name:
    foundry_api_key = notebookutils.credentials.getSecret(key_vault_uri, foundry_key_secret_name)
else:
    foundry_api_key = ""

print(f"Deployment: {chat_deployment}")
print(f"API version: {api_version}")
print(f"Authentication: {'API key from secure configuration' if foundry_api_key else 'Entra token'}")

In [ ]:
appeals = spark.table(source_table)
required_columns = {"appeal_id", "narrative", "reason_code", "ground_truth_sentiment", "synthetic"}
missing_columns = required_columns.difference(appeals.columns)
if missing_columns:
    raise ValueError(f"Missing required columns in {source_table}: {sorted(missing_columns)}")
if appeals.filter(F.col("synthetic") != True).count() != 0:
    raise ValueError("Foundry enrichment is restricted to synthetic appeal records in this lab.")
print(f"Appeals available for enrichment: {appeals.count()}")

In [ ]:
chat_url = (
    f"{foundry_endpoint}/openai/deployments/{chat_deployment}"
    f"/chat/completions?api-version={api_version}"
)

def auth_headers():
    if foundry_api_key:
        return {"api-key": foundry_api_key, "Content-Type": "application/json"}
    token = notebookutils.credentials.getToken("https://cognitiveservices.azure.com")
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

allowed_sentiments = {"positive", "neutral", "negative"}
allowed_reasons = {
    "COMPARABLE_SALES", "PROPERTY_CONDITION", "DATA_CORRECTION",
    "CLASSIFICATION", "RENOVATION_TIMING", "INFORMATION_REQUEST"
}

system_prompt = """
You analyze synthetic property-assessment appeal narratives. Return one JSON object with exactly these fields:
- sentiment: positive, neutral, or negative
- summary: one factual sentence grounded only in the narrative
- reasonCode: COMPARABLE_SALES, PROPERTY_CONDITION, DATA_CORRECTION, CLASSIFICATION, RENOVATION_TIMING, or INFORMATION_REQUEST
- followUpRequired: boolean indicating whether the narrative requests prompt human review
Do not infer identity, protected characteristics, legal outcomes, valuation decisions, or facts absent from the narrative.
""".strip()

def enrich_appeal(row):
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row.narrative},
        ],
        "max_completion_tokens": 350,
        "response_format": {"type": "json_object"},
    }
    response = requests.post(chat_url, headers=auth_headers(), json=payload, timeout=120)
    response.raise_for_status()
    result = json.loads(response.json()["choices"][0]["message"]["content"])
    sentiment = str(result["sentiment"]).lower()
    reason_code = str(result["reasonCode"]).upper()
    summary = str(result["summary"]).strip()
    follow_up = result["followUpRequired"]
    if sentiment not in allowed_sentiments:
        raise ValueError(f"Unexpected sentiment for {row.appeal_id}: {sentiment}")
    if reason_code not in allowed_reasons:
        raise ValueError(f"Unexpected reason code for {row.appeal_id}: {reason_code}")
    if not summary or not isinstance(follow_up, bool):
        raise ValueError(f"Invalid summary or follow-up value for {row.appeal_id}")
    return (row.appeal_id, sentiment, summary, reason_code, "urgent_follow_up" if follow_up else "standard_review")


In [ ]:
rows = appeals.select("appeal_id", "narrative").collect()
if max_appeals > 0:
    rows = rows[:max_appeals]

with ThreadPoolExecutor(max_workers=max_workers) as pool:
    results = list(pool.map(enrich_appeal, rows))

result_schema = StructType([
    StructField("appeal_id", StringType(), False),
    StructField("ai_sentiment", StringType(), False),
    StructField("ai_summary", StringType(), False),
    StructField("ai_reason_code", StringType(), False),
    StructField("ai_follow_up", StringType(), False),
])
signals = spark.createDataFrame(results, schema=result_schema)
enriched = (appeals.join(signals, "appeal_id", "inner")
    .withColumn("enriched_at_utc", F.current_timestamp())
    .withColumn("ai_provider", F.lit("Microsoft Foundry"))
    .withColumn("ai_deployment", F.lit(chat_deployment)))

if enriched.count() != len(results):
    raise ValueError("Enrichment output did not preserve one row per processed appeal.")
(enriched.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(target_table))
print(f"Wrote {enriched.count()} enriched appeals to {target_table}")

In [ ]:
evaluation = (spark.table(target_table)
    .withColumn("sentiment_match", F.lower("ai_sentiment") == F.lower("ground_truth_sentiment"))
    .withColumn("reason_match", F.upper("ai_reason_code") == F.upper("reason_code")))

display(evaluation.groupBy("ground_truth_sentiment", "ai_sentiment").count())
display(evaluation.agg(
    F.count("*").alias("records"),
    F.avg(F.col("sentiment_match").cast("double")).alias("sentiment_accuracy"),
    F.avg(F.col("reason_match").cast("double")).alias("reason_code_accuracy"),
))